<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Distributions
days_since_last_update, impressions_90d, and avg_position are right-skewed, with means above medians and long upper tails. impressions_90d is particularly heavy-tailed (median 731 vs. maximum 517,715). ctr is also highly skewed, with a median of 0.07 and extreme upper values. Therefore, medians and percentile-based buckets are more informative than means for these signals.

In [9]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/thahsinj06/Fly-rank-ml-internship-work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

signals = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

print("Dataset shape:", df.shape)
print("\nRequired signals:")

for col in signals:
    print(f"{'✓' if col in df.columns else '✗'} {col}")

print("\nDistribution summary:")

for col in signals:
    s = df[col].dropna()

    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")
    print(f"n      : {len(s):,}")
    print(f"missing: {df[col].isna().sum():,}")
    print(f"min    : {s.min():,.4f}")
    print(f"median : {s.median():,.4f}")
    print(f"mean   : {s.mean():,.4f}")
    print(f"p90    : {s.quantile(0.90):,.4f}")
    print(f"p99    : {s.quantile(0.99):,.4f}")
    print(f"max    : {s.max():,.4f}")

Dataset shape: (30000, 44)

Required signals:
✓ days_since_last_update
✓ impressions_90d
✓ avg_position
✓ ctr

Distribution summary:

days_since_last_update
n      : 30,000
missing: 0
min    : 1.0000
median : 20.0000
mean   : 46.0983
p90    : 104.0000
p99    : 106.0000
max    : 373.0000

impressions_90d
n      : 30,000
missing: 0
min    : 1.0000
median : 731.0000
mean   : 5,200.3663
p90    : 12,136.4000
p99    : 73,505.8300
max    : 517,715.0000

avg_position
n      : 30,000
missing: 0
min    : 0.0000
median : 10.8000
mean   : 16.3424
p90    : 36.8000
p99    : 69.9010
max    : 245.0000

ctr
n      : 30,000
missing: 0
min    : 0.0000
median : 0.0700
mean   : 0.5107
p90    : 0.6500
p99    : 8.3300
max    : 100.0000


## 2. Signal tests

Three candidate signals were tested against the observed decline direction. The tests are directional and are used for decision-support rather than causal inference.

- **Staleness — MIXED:** decline rates increased from 0–30 through 91–180 days, but the 180+ bucket reversed. The smallest buckets also have limited sample sizes.
- **Search volume — CONFIRMED:** low-volume pages had a substantially lower decline rate than the higher-volume groups, although the highest-volume bucket dipped slightly.
- **CTR vs position — MIXED:** CTR did not show a clean monotonic pattern across position buckets, and the earlier distribution check showed extreme CTR values. CTR should therefore be treated cautiously..*

In [10]:
# Section 2 — Signal tests

df["_declining"] = (df["trend_direction"] == "down").astype(int)

# ------------------------------------------------------------
# Signal 1 — Staleness
# ------------------------------------------------------------

df["_staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, np.inf],
    labels=["0–30 days", "31–90 days", "91–180 days", "180+ days"],
    include_lowest=True
)

staleness_test = (
    df.groupby("_staleness_bucket", observed=False)["_declining"]
      .agg(n="size", decline_rate="mean")
      .reset_index()
)

staleness_test["decline_rate"] *= 100

print("=" * 60)
print("SIGNAL 1 — STALENESS")
print("=" * 60)
print(staleness_test.to_string(index=False))

# ------------------------------------------------------------
# Signal 2 — Search volume
# ------------------------------------------------------------

df["_volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

volume_test = (
    df.groupby("_volume_bucket", observed=False)["_declining"]
      .agg(n="size", decline_rate="mean")
      .reset_index()
)

volume_test["decline_rate"] *= 100

print("\n" + "=" * 60)
print("SIGNAL 2 — SEARCH VOLUME")
print("=" * 60)
print(volume_test.to_string(index=False))

# ------------------------------------------------------------
# Signal 3 — CTR vs Position
# ------------------------------------------------------------

df["_position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["Top 3", "4–10", "11–20", "20+"]
)

ctr_position_test = (
    df.groupby("_position_bucket", observed=False)["ctr"]
      .agg(
          n="size",
          median_ctr="median",
          mean_ctr="mean"
      )
      .reset_index()
)

print("\n" + "=" * 60)
print("SIGNAL 3 — CTR vs POSITION")
print("=" * 60)
print(ctr_position_test.to_string(index=False))

SIGNAL 1 — STALENESS
_staleness_bucket     n  decline_rate
        0–30 days 20480     51.137695
       31–90 days   175     58.857143
      91–180 days  9171     61.105659
        180+ days   174     47.126437

SIGNAL 2 — SEARCH VOLUME
_volume_bucket    n  decline_rate
           Low 7503     37.611622
        Medium 7499     60.461395
          High 7498     62.563350
     Very High 7500     56.200000

SIGNAL 3 — CTR vs POSITION
_position_bucket     n  median_ctr  mean_ctr
           Top 3  2346        0.00  1.472869
            4–10 11842        0.16  0.651045
           11–20  7273        0.10  0.323443
             20+  8539        0.00  0.211333


## 3. The flag-linked test

Staleness is one of the signals behind FlyRank's refresh logic. Pages that had not been updated for 90+ days showed a 60.85% observed decline rate, compared with 51.14% for pages updated within 30 days. This is a 9.71 percentage-point difference and provides directional support for using staleness as a refresh signal, although it does not establish causation.

In [11]:
# Section 3 — Flag-linked test: refresh / staleness

# Compare recently updated pages with clearly stale pages.
df["_refresh_staleness_group"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, np.inf],
    labels=["Recently updated (0–30)", "Aging (31–90)", "Stale (90+)"],
    include_lowest=True
)

flag_linked_test = (
    df.groupby("_refresh_staleness_group", observed=False)["_declining"]
      .agg(
          n="size",
          decline_rate="mean"
      )
      .reset_index()
)

flag_linked_test["decline_rate"] *= 100

print("=" * 60)
print("FLAG-LINKED TEST — REFRESH / STALENESS")
print("=" * 60)
print(flag_linked_test.to_string(index=False))

# Compare stale pages directly against recently updated pages
recent_rate = flag_linked_test.loc[
    flag_linked_test["_refresh_staleness_group"] == "Recently updated (0–30)",
    "decline_rate"
].iloc[0]

stale_rate = flag_linked_test.loc[
    flag_linked_test["_refresh_staleness_group"] == "Stale (90+)",
    "decline_rate"
].iloc[0]

difference = stale_rate - recent_rate

print(f"\nDecline-rate difference (Stale 90+ vs Recent 0–30): {difference:.2f} percentage points")

FLAG-LINKED TEST — REFRESH / STALENESS
_refresh_staleness_group     n  decline_rate
 Recently updated (0–30) 20480     51.137695
           Aging (31–90)   175     58.857143
             Stale (90+)  9345     60.845372

Decline-rate difference (Stale 90+ vs Recent 0–30): 9.71 percentage points


## 4. What this means in practice

The observed data suggests that content teams can use staleness as a directional signal when prioritising pages for refresh, since pages older than 90 days showed a higher decline rate than recently updated pages. Search volume also appears useful for prioritisation, while CTR should be treated more cautiously because of its skewed distribution and inconsistent pattern across position buckets. These signals should support human review rather than be treated as proof that a page needs a specific action.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.